# Notebook 3: Radar/Lidar Point Cloud Attacks

Demonstrate all 6 point cloud attack types on Radar and Lidar.

**Attacks:** Ghost Injection, Cluster Split, Cluster Merge, Point Suppression, Noise Floor, Random Perturbation

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from data_loader import SensorDataLoader
from attacks.radar_lidar_attacks import PointCloudAttacker, PointCloudAttackType

%matplotlib inline

## 3.1 Load Data

In [ ]:
SCENARIO = 'scenario2'
DATA_PATH = f'../data/sensor_fusion_dataset/{SCENARIO}'

loader = SensorDataLoader(DATA_PATH)
detections = loader.load_all_detections()

lidar = detections[1]
radar = detections[2]

print(f"Lidar: {len(lidar)} detections")
print(f"Radar: {len(radar)} detections")

## 3.2 Initialize Attacker

In [ ]:
attacker = PointCloudAttacker()
print("Available attacks:", [a.name for a in PointCloudAttackType])

## 3.3 Run Attacks on Lidar

In [ ]:
attacks = [
    PointCloudAttackType.GHOST_INJECTION,
    PointCloudAttackType.CLUSTER_SPLIT,
    PointCloudAttackType.CLUSTER_MERGE,
    PointCloudAttackType.POINT_SUPPRESSION,
    PointCloudAttackType.NOISE_FLOOR,
    PointCloudAttackType.RANDOM_PERTURBATION
]

lidar_results = {}
for attack in attacks:
    attacked = attacker.attack_detections(lidar.copy(), attack, sensor_id=1)
    lidar_results[attack.name] = attacked
    print(f"{attack.name}: {len(attacked)} detections (delta: {len(attacked) - len(lidar)})")

## 3.4 Visualize Point Cloud Attacks

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (attack_name, attacked_df) in enumerate(lidar_results.items()):
    ax = axes[idx]
    
    # Plot benign
    ax.scatter(lidar['x_piren'], lidar['y_piren'], s=3, alpha=0.3, c='blue', label='Benign')
    
    # Plot attacked (only changed/new points in red)
    ax.scatter(attacked_df['x_piren'], attacked_df['y_piren'], s=3, alpha=0.3, c='red', label='Attacked')
    
    ax.set_title(f'{attack_name}\n{len(attacked_df)} detections')
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.legend()
    ax.grid(True)
    ax.set_aspect('equal')

plt.suptitle('Lidar Point Cloud Attacks', fontsize=14)
plt.tight_layout()
plt.show()

## 3.5 Ghost Injection Detail

In [ ]:
# Show ghost injection with different numbers of ghosts
ghost_counts = [5, 10, 20, 50]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, n_ghosts in enumerate(ghost_counts):
    ax = axes[idx]
    
    # Temporarily modify attacker
    att = PointCloudAttacker()
    attacked = att.attack_detections(lidar.copy(), PointCloudAttackType.GHOST_INJECTION, sensor_id=1)
    
    ax.scatter(lidar['x_piren'], lidar['y_piren'], s=5, alpha=0.5, c='blue', label='Original')
    ax.scatter(attacked['x_piren'], attacked['y_piren'], s=5, alpha=0.3, c='red', label='With Ghosts')
    
    ax.set_title(f'Ghost Injection\nTotal: {len(attacked)} detections')
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.legend()
    ax.grid(True)
    ax.set_aspect('equal')

plt.suptitle('Ghost Injection Attack Detail', fontsize=14)
plt.tight_layout()
plt.show()

## 3.6 Radar vs Lidar Attack Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for col, attack in enumerate(attacks[:3]):
    # Lidar
    att_lidar = attacker.attack_detections(lidar.copy(), attack, sensor_id=1)
    ax = axes[0, col]
    ax.scatter(lidar['x_piren'], lidar['y_piren'], s=3, alpha=0.3, c='blue')
    ax.scatter(att_lidar['x_piren'], att_lidar['y_piren'], s=3, alpha=0.3, c='red')
    ax.set_title(f'Lidar - {attack.name}')
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.grid(True)
    ax.set_aspect('equal')
    
    # Radar
    att_radar = attacker.attack_detections(radar.copy(), attack, sensor_id=2)
    ax = axes[1, col]
    ax.scatter(radar['x_piren'], radar['y_piren'], s=3, alpha=0.3, c='blue')
    ax.scatter(att_radar['x_piren'], att_radar['y_piren'], s=3, alpha=0.3, c='red')
    ax.set_title(f'Radar - {attack.name}')
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.grid(True)
    ax.set_aspect('equal')

plt.suptitle('Point Cloud Attacks: Lidar vs Radar', fontsize=14)
plt.tight_layout()
plt.show()